# FEWS OAuth2 test vanuit .env

Dit notebook volgt je voorbeeldscript en laadt variabelen uit `.env` (en daarna `.env.development` als fallback).

Benodigde variabelen in je `.env`:
- `FEWSPY_TEST_FEWS_URL`
- `FEWSPY_TEST_OAUTH2_TOKEN_URL`
- `FEWSPY_TEST_OAUTH2_CLIENT_ID`
- `FEWSPY_TEST_OAUTH2_CLIENT_SECRET`
- `FEWSPY_TEST_OAUTH2_SCOPE`

Optioneel:
- `FEWSPY_TEST_OAUTH2_CERT` (pad naar PEM)
- `FEWSPY_TEST_OAUTH2_VERIFY` (`true`/`false` of pad naar CA bundle)
- `FEWSPY_TEST_USE_TEMP_TOKEN` (`true`/`false`)
- `FEWSPY_TEST_ACCESS_TOKEN` (alleen nodig als `FEWSPY_TEST_USE_TEMP_TOKEN=true`)

In [1]:
import os
from pathlib import Path

import requests
from requests.auth import HTTPBasicAuth


def load_dotenv_file(path: Path) -> None:
    if not path.exists():
        return

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if (not line) or line.startswith("#") or ("=" not in line):
            continue

        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('\"').strip("'"))


# Eerst .env, daarna .env.development (alleen vullen als key nog niet bestaat)
load_dotenv_file(Path.cwd() / ".env")
load_dotenv_file(Path.cwd() / ".env.development")

USE_TEMP_TOKEN = os.getenv("FEWSPY_TEST_USE_TEMP_TOKEN", "false").strip().lower() in {"1", "true", "yes", "on"}
TEMP_ACCESS_TOKEN = os.getenv("FEWSPY_TEST_ACCESS_TOKEN")

if USE_TEMP_TOKEN:
    required = ["FEWSPY_TEST_FEWS_URL", "FEWSPY_TEST_ACCESS_TOKEN"]
else:
    required = [
        "FEWSPY_TEST_FEWS_URL",
        "FEWSPY_TEST_OAUTH2_TOKEN_URL",
        "FEWSPY_TEST_OAUTH2_CLIENT_ID",
        "FEWSPY_TEST_OAUTH2_CLIENT_SECRET",
        "FEWSPY_TEST_OAUTH2_SCOPE",
    ]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise ValueError(f"Missende env variabelen: {', '.join(missing)}")

FEWS_URL = os.environ["FEWSPY_TEST_FEWS_URL"].rstrip("/") + "/"
TOKEN_URL = os.getenv("FEWSPY_TEST_OAUTH2_TOKEN_URL")
CLIENT_ID = os.getenv("FEWSPY_TEST_OAUTH2_CLIENT_ID")
CLIENT_SECRET = os.getenv("FEWSPY_TEST_OAUTH2_CLIENT_SECRET")
SCOPE = os.getenv("FEWSPY_TEST_OAUTH2_SCOPE")
CERT = os.getenv("FEWSPY_TEST_OAUTH2_CERT")
VERIFY_RAW = os.getenv("FEWSPY_TEST_OAUTH2_VERIFY", "true").strip()

if VERIFY_RAW.lower() in {"true", "1", "yes", "on"}:
    VERIFY = True
elif VERIFY_RAW.lower() in {"false", "0", "no", "off"}:
    VERIFY = False
else:
    # Als het geen bool-string is, behandelen als CA-bundle pad
    VERIFY = VERIFY_RAW

print(".env geladen. Basisconfig aanwezig.")
print(f"FEWS endpoint: {FEWS_URL}")
print(f"Token endpoint: {TOKEN_URL}")
print(f"Cert ingesteld: {bool(CERT)}")
print(f"Gebruik tijdelijke token: {USE_TEMP_TOKEN}")
print(f"TLS verify type: {type(VERIFY).__name__}")

.env geladen. Basisconfig aanwezig.
FEWS endpoint: https://fewsapi.hhnk.nl/FewsWebServices/rest/fewspiservice/v1/
Token endpoint: https://login.microsoftonline.com/d2e6706d-08b5-4123-bea1-186399f9d47f/oauth2/v2.0/token
Cert ingesteld: True
Gebruik tijdelijke token: True
TLS verify type: bool


In [2]:
# 1) Access token bepalen (tijdelijke token of OAuth2 ophalen)
if USE_TEMP_TOKEN:
    if not TEMP_ACCESS_TOKEN:
        raise ValueError("FEWSPY_TEST_ACCESS_TOKEN ontbreekt terwijl FEWSPY_TEST_USE_TEMP_TOKEN=true is")
    access_token = TEMP_ACCESS_TOKEN
    print("Tijdelijke access token uit .env gebruikt")
else:
    token_data = {"grant_type": "client_credentials", "scope": SCOPE}

    token_response = requests.post(
        TOKEN_URL,
        data=token_data,
        auth=HTTPBasicAuth(CLIENT_ID, CLIENT_SECRET),
        cert=CERT if CERT else None,
        verify=VERIFY,
        timeout=30,
    )

    print("Token status:", token_response.status_code)
    if token_response.status_code >= 400:
        print("Token body:", token_response.text)
    token_response.raise_for_status()
    access_token = token_response.json()["access_token"]
    print("Access token ontvangen")

Tijdelijke access token uit .env gebruikt


In [4]:
# 2) FEWS timeseries request
timeseries_url = FEWS_URL + "timeseries/"
headers = {"Authorization": f"Bearer {access_token}"}

params = {
    "documentFormat": "PI_XML",
    "documentVersion": "1.34",
    "parameterIds": "Q.meting",
    "locationIds": "MPN-E-1071",
    "startTime": "2025-07-02T12:44:53Z",
    "endTime": "2025-07-02T13:44:53Z",
    "convertDatum": "true",
}

result = requests.get(
    timeseries_url,
    params=params,
    headers=headers,
    verify=VERIFY,
    timeout=60,
    cert=CERT,
)

print("Timeseries status:", result.status_code)
if (result.status_code == 401) and ("Expired JWT" in result.text):
    raise RuntimeError("De opgegeven access token is verlopen (Expired JWT).")
result.raise_for_status()

print("Content-Type:", result.headers.get("Content-Type"))
print("Eerste 500 bytes:")
print(result.content[:500])

Timeseries status: 200
Content-Type: application/xml
Eerste 500 bytes:
b'<?xml version="1.0" encoding="UTF-8"?>\n<TimeSeries xmlns="http://www.wldelft.nl/fews/PI" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.wldelft.nl/fews/PI https://fewsdocs.deltares.nl/schemas/version1.0/pi-schemas/pi_timeseries.xsd" version="1.34" xmlns:fs="http://www.wldelft.nl/fews/fs">\n    <timeZone>0.0</timeZone>\n    <series>\n        <header>\n            <type>instantaneous</type>\n            <moduleInstanceId>Productie</moduleInstanceId>\n            <lo'
